# Building thermal modelling — Question 2: a family of R-C models

In Question 1 we explored the data. Now we build models that predict the **indoor temperature** from the weather and the
heating power, and we make them in three sizes, from very simple to fairly detailed. We train them on **year 1** and
(in Question 3) test them on **year 2**.

> **How can we build R-C models of different complexity, and what does each one learn?**

This is a standalone notebook: it reloads the data and the plotting helper first, then builds the models. Figures are
saved to `../figures/`.

## What is an R-C model?

A building heats up and cools down a lot like a simple electrical circuit. The analogy is the reason these are called
**R-C (resistor–capacitor) models**:

- a **capacitor (C)** stores charge ⟷ the building stores **heat** (its walls, floor, furniture, air);
- a **resistor (R)** limits current flow ⟷ insulation limits **heat flow** between inside and outside.

So a temperature is like a voltage, a heat flow is like a current, and "thermal mass" is like capacitance. With one or
two of these C's and a few R's we can describe how the indoor temperature reacts to the outdoor temperature, the heating,
and the sun. The models are simple, but their parameters could mean something physical, which makes them easy to check.

## Outline

1. [Setup and data](#sec-setup)
2. [How we train the models](#sec-method)
3. [Model A — one resistor, one capacitor (1R1C)](#sec-A)
4. [Model B — adding the walls (2R2C)](#sec-B)
5. [Model C — adding the sun (3R2C)](#sec-C)
6. [The code: building, discretising and fitting](#sec-code)
7. [Fitted parameters — are they physically sensible?](#sec-params)
8. [Checking the fit on year 1](#sec-fit)
9. [Which parameters can we actually trust? (identifiability)](#sec-ident)
10. [Answer to Question 2](#sec-answer)

<a id="sec-setup"></a>
## 1. Setup and data

We reload the dataset and rebuild the same helper columns and `save_fig` function used in the Question 1 notebook, so this notebook runs on its own.

In [7]:
import os
import numpy as np                     # arrays and maths
import pandas as pd                    # tables
import plotly.graph_objects as go      # plotting
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "plotly_white"

FIG_DIR = "../figures"
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(fig, name, h=420):
    # save an interactive .html and a static .png, and return the figure so it shows in the notebook
    fig.update_layout(height=h, margin=dict(l=60, r=30, t=70, b=50))
    fig.write_html(f"{FIG_DIR}/{name}.html", include_plotlyjs="cdn")
    try:
        fig.write_image(f"{FIG_DIR}/{name}.png", scale=2)
    except Exception as e:
        print(f"[warn] could not save PNG for {name}: {e}")
    return fig

In [8]:
# Load the data and split the two years by row position (first 35040 rows = year 1). Temperatures -> °C.
df = pd.read_csv("../data/interview_data_buildings/interview_data_buildings.csv", index_col=0)
df["year"] = np.where(np.arange(len(df)) < 35040, 1, 2)
for col in ["indoor_temperature", "outside_temperature"]:
    df[col + "_C"] = df[col] - 273.15
print("Loaded", len(df), "rows |", (df.year == 1).sum(), "in year 1,", (df.year == 2).sum(), "in year 2")

Loaded 70080 rows | 35040 in year 1, 35040 in year 2


<a id="sec-method"></a>
## 2. How we train the models

**The plan.** We only ever use **year 1** to choose the model parameters. Year 2 is kept aside for testing
(Question 3). We do **not** shuffle the data into random train/test pieces, this is a time series, and shuffling would
let the model "peek" at the future. Training on the past and testing on the future is the honest setup.

**Continuous-time description.** Each model is a small set of differential equations of the form
$$\dot{\mathbf{x}} = \mathbf{A}\,\mathbf{x} + \mathbf{B}\,\mathbf{u},$$
where $\mathbf{x}$ holds the temperatures the model tracks and $\mathbf{u}$ holds the inputs (outdoor temperature,
heating power, and for Model C the sun). Writing the model this way means the parameters (the R's and C's) are real
physical quantities and do not depend on the 15-minute sampling.

**Turning it into time steps (discretisation).** To run the model on 15-minute data we convert the differential
equations into a step-by-step update $\mathbf{x}_{k+1} = \mathbf{F}\,\mathbf{x}_k + \mathbf{G}\,\mathbf{u}_k$. The two
matrices play different roles: $\mathbf{F} = e^{\mathbf{A}\,\Delta t}$ says how the tracked temperatures evolve *on
their own* over one step — how much of the current state "survives" 15 minutes of drifting towards the surroundings —
while $\mathbf{G} = \mathbf{A}^{-1}\left(\mathbf{F} - \mathbf{I}\right)\mathbf{B}$ says how strongly the inputs
(weather, heating, sun), held constant over the step, push those temperatures during the same 15 minutes. Both come
from the *exact* zero-order-hold solution of the differential equations (via the matrix exponential) instead of a rough
approximation, which keeps the slow thermal behaviour accurate.

**Why not just run the model freely over the whole year and fit that?**
<details><summary>Click for answer</summary>

Because the house has a thermostat. The recorded heating power is what that thermostat decided to do to keep the room
near a set temperature. If we let our model run free over a whole year on this heating signal, a tiny error in the
parameters slowly pushes the predicted temperature far from reality (there is no feedback to pull it back). We checked:
a naive "free-run" fit collapses to a useless near-flat line or blows up.

The standard fix, which we use, is **multiple shooting**: every so often we reset the model to the measured temperature
and let it predict forward only a limited time (here we mix three windows: one step = 30 min, one day, and three days).
We then choose the parameters that make these short forecasts as accurate as possible. This is both stable and exactly
the kind of forecasting we test in Question 3.
</details>

**Keeping the parameters physical.** Resistances and capacitances must be positive, so we fit their logarithms
(which can never go negative); the same trick keeps Model C's solar window area between 0.01 and 30 m². The fit itself
is a least-squares optimisation.

<a id="sec-A"></a>
## 3. Model A — one resistor, one capacitor (1R1C)

The simplest possible model: treat the **whole house as a single lump** with heat capacity $C_i$, connected to
the outside through a single resistance $R_i$, and add the heating power directly.

**Tracks:** indoor temperature $T_i$. **Inputs:** outdoor temperature $T_o$, heating power $\Phi_h$.
$$C_i\,\dot T_i = \underbrace{\frac{T_o - T_i}{R_i}}_{\text{heat lost to outside}} + \underbrace{\Phi_h}_{\text{heating}}$$

It has a single "speed" (time constant $\tau = R_i C_i$) and a single overall heat-loss number $UA = 1/R_i$ (watts lost
per degree of temperature difference). It cannot tell the slow walls apart from the fast indoor air, and it has no sun
term.

<a id="sec-B"></a>
## 4. Model B — adding the walls (2R2C)

Real buildings have two speeds: the air reacts quickly, the heavy walls slowly. Model B adds a **second lump**
$T_e$ for the walls/structure (we never measure it directly — the model works it out). Heat flows air↔walls through
$R_{ie}$ and walls↔outside through $R_{ea}$.

**Tracks:** $T_i$ (air) and $T_e$ (walls). **Inputs:** $T_o$, $\Phi_h$.
$$C_i\,\dot T_i = \frac{T_e - T_i}{R_{ie}} + \Phi_h, \qquad
  C_e\,\dot T_e = \frac{T_i - T_e}{R_{ie}} + \frac{T_o - T_e}{R_{ea}}$$
This captures the fast and slow behaviour we saw in the "memory" plot of Question 1. Overall heat loss is
$UA = 1/(R_{ie}+R_{ea})$ (the two resistors in series). Still no sun term.

<a id="sec-C"></a>
## 5. Model C — adding the sun (3R2C)

The most detailed model. It keeps the two lumps and adds:

- a **third resistance** $R_{ia}$ — a direct, fast path from inside to outside (windows, draughts, ventilation);
- a **solar term** $A_s$ — an effective window area that lets sunshine $\Phi_s$ warm the room (this is the piece A and B
  are missing, and it is what mattered in summer in Question 1).

**Tracks:** $T_i$, $T_e$. **Inputs:** $T_o$, $\Phi_h$, $\Phi_s$.
$$C_i\,\dot T_i = \frac{T_e - T_i}{R_{ie}} + \frac{T_o - T_i}{R_{ia}} + \Phi_h + A_s\,\Phi_s, \qquad
  C_e\,\dot T_e = \frac{T_i - T_e}{R_{ie}} + \frac{T_o - T_e}{R_{ea}}$$
Overall heat loss is now $UA = 1/R_{ia} + 1/(R_{ie}+R_{ea})$ (the direct path in parallel with the wall path). The price
of the extra realism is two extra parameters, and we will have to check whether the data can pin them all down (Section 9).

<a id="sec-code"></a>
## 6. The code: building, discretising and fitting

The cell below contains the whole machinery: it builds the matrices for each model, converts them to 15-minute steps, runs them quickly (sped up with `numba`), and fits the parameters with the multiple-shooting method described above. Fitting all three models takes only a few seconds, so the notebook simply does it directly.

In [9]:
from scipy.linalg import expm            # matrix exponential, used for the exact discretisation
from scipy.optimize import least_squares   # the optimiser that fits the parameters
from numba import njit                      # compiles the simulation loop to make fitting fast

# --- training data: year 1, at the native 15-minute resolution ---
d1 = df[df.year == 1]
Ti_tr = d1["indoor_temperature_C"].values
src_tr = {"To": d1["outside_temperature_C"].values,
          "Qh": d1["heating_power"].values,
          "Qs": d1["global_horizontal_solar_radiation"].values}
DT = 900.0                       # one time step = 900 s = 15 min
HSET = [2, 96, 288]              # forecast windows used while fitting: 1 step (30 min), 1 day, 3 days

def matrices(model, p):
    # Build the continuous-time matrices A (state) and B (inputs) for the chosen model from a dict of parameters p.
    if model == "A":                                          # 1R1C : tracks [Ti] ; inputs [To, Qh]
        Ri, Ci = p["Ri"], p["Ci"]
        A = np.array([[-1/(Ri*Ci)]])
        B = np.array([[1/(Ri*Ci), 1/Ci]])
        cols = ("To", "Qh")
    elif model == "B":                                        # 2R2C : tracks [Ti, Te] ; inputs [To, Qh]
        Rie, Rea, Ci, Ce = p["Rie"], p["Rea"], p["Ci"], p["Ce"]
        A = np.array([[-1/(Rie*Ci),               1/(Rie*Ci)],
                      [ 1/(Rie*Ce), -1/(Rie*Ce) - 1/(Rea*Ce)]])
        B = np.array([[0.0,        1/Ci],
                      [1/(Rea*Ce), 0.0 ]])
        cols = ("To", "Qh")
    else:                                                     # 3R2C : tracks [Ti, Te] ; inputs [To, Qh, Qs]
        Rie, Ria, Rea, Ci, Ce = p["Rie"], p["Ria"], p["Rea"], p["Ci"], p["Ce"]
        As = p["As"]
        A = np.array([[-1/(Rie*Ci) - 1/(Ria*Ci),  1/(Rie*Ci)],
                      [ 1/(Rie*Ce),               -1/(Rie*Ce) - 1/(Rea*Ce)]])
        B = np.array([[1/(Ria*Ci), 1/Ci, As/Ci],
                      [1/(Rea*Ce), 0.0,  0.0  ]])
        cols = ("To", "Qh", "Qs")
    return A, B, cols

def discretize(A, B, dt):
    # Exact "zero-order hold" discretisation: F = exp(A*dt), G = A^{-1}(F - I) B  (series fallback if A is singular).
    F = expm(A * dt)
    try:
        G = np.linalg.solve(A, (F - np.eye(A.shape[0])) @ B)
    except np.linalg.LinAlgError:
        I = np.eye(A.shape[0]); S = I*dt; term = I*dt
        for k in range(1, 12):
            term = term @ A * dt / (k+1); S = S + term
        G = S @ B
    return F, G

@njit(cache=False)
def _freerun(F, G, U, x0, n, ns):
    # Run the model forward with no feedback: x_{k+1} = F x_k + G u_k.
    X = np.zeros((n, ns)); X[0] = x0
    for t in range(1, n):
        X[t] = F @ X[t-1] + G @ U[t-1]
    return X

@njit(cache=False)
def _ms_resid(F, G, U, Tm, H, n, ns):
    # Multiple-shooting residuals: reset to the measured temperature every H steps, predict H ahead, compare.
    res = np.zeros(n); nseg = n // H
    for sg in range(nseg):
        s = sg*H; x = np.zeros(ns)
        for k in range(ns): x[k] = Tm[s]      # reset all tracked temperatures to the measured indoor temp
        for t in range(1, H):
            x = F @ x + G @ U[s+t-1]
            res[s+t] = x[0] - Tm[s+t]          # error between predicted and measured indoor temperature
    return res

@njit(cache=False)
def _h_rmse(F, G, U, Tm, H, n, ns):
    # RMSE of an H-step-ahead forecast (reset to measurement every H steps).
    se = 0.0; c = 0; nseg = n // H
    for sg in range(nseg):
        s = sg*H; x = np.zeros(ns)
        for k in range(ns): x[k] = Tm[s]
        for t in range(1, H):
            x = F @ x + G @ U[s+t-1]; se += (x[0]-Tm[s+t])**2; c += 1
    return (se/c)**0.5

# parameter names per model, and sensible starting guesses for a ~150 m² UK house
PARAM_NAMES = {"A": ["Ri", "Ci"],
               "B": ["Rie", "Rea", "Ci", "Ce"],
               "C": ["Rie", "Ria", "Rea", "Ci", "Ce", "As"]}
P0 = {"A": dict(Ri=4e-3, Ci=1.5e7),
      "B": dict(Rie=8e-4, Rea=4e-3, Ci=8e6, Ce=4e7),
      "C": dict(Rie=8e-4, Ria=6e-3, Rea=4e-3, Ci=8e6, Ce=4e7, As=4.0)}

# physical ranges (we fit log R, log C and log As, so all parameters stay positive and inside a sensible band)
R_LO, R_HI = np.log(5e-4), np.log(3e-2)
C_LO, C_HI = np.log(1e6),  np.log(1e8)
AS_LO, AS_HI = np.log(0.01), np.log(30.0)
def _isR(n): return n in ("Ri", "Rie", "Ria", "Rea")
def _isC(n): return n in ("Ci", "Ce")

def vec_to_params(model, z):
    # turn the optimiser's vector of log-parameters back into real, positive parameters
    return {nme: np.exp(z[i]) for i, nme in enumerate(PARAM_NAMES[model])}

def params_to_vec(model, p):
    return np.array([np.log(p[nme]) for nme in PARAM_NAMES[model]], float)

def bounds(model):
    lo, hi = [], []
    for nme in PARAM_NAMES[model]:
        if   _isR(nme):   lo.append(R_LO);  hi.append(R_HI)
        elif _isC(nme):   lo.append(C_LO);  hi.append(C_HI)
        else:             lo.append(AS_LO); hi.append(AS_HI)   # the solar window area
    return np.array(lo), np.array(hi)

def U_of(cols, src): return np.ascontiguousarray(np.column_stack([src[c] for c in cols]))

def simulate_freerun(model, p, src, Ti0, dt):
    # convenience wrapper: build -> discretise -> run free, return the predicted indoor temperature
    A, B, cols = matrices(model, p); F, G = discretize(A, B, dt)
    ns = A.shape[0]; U = U_of(cols, src)
    return _freerun(F, G, U, np.full(ns, Ti0), len(U), ns)[:, 0]

def fit(model):
    # Fit one model by minimising the multiple-shooting forecast error over all three windows in HSET.
    A0, _, cols = matrices(model, P0[model]); ns = A0.shape[0]
    U = U_of(cols, src_tr)
    def resid(z):
        A, B, _ = matrices(model, vec_to_params(model, z)); F, G = discretize(A, B, DT)
        return np.concatenate([_ms_resid(F, G, U, Ti_tr, H, len(Ti_tr), ns) for H in HSET])
    lo, hi = bounds(model)
    sol = least_squares(resid, params_to_vec(model, P0[model]), bounds=(lo, hi),
                        method="trf", x_scale="jac", max_nfev=4000)
    p = vec_to_params(model, sol.x)
    # uncertainty of each parameter (from the optimiser's Jacobian) + whether it ended up stuck at a limit
    r = sol.fun; dof = max(len(r) - len(sol.x), 1); sigma2 = float(r @ r) / dof
    rel_std, at_bound = {}, {}
    try:
        cov = sigma2 * np.linalg.inv(sol.jac.T @ sol.jac); cond = float(np.linalg.cond(sol.jac.T @ sol.jac))
        for i, nme in enumerate(PARAM_NAMES[model]):
            zp = sol.x.copy(); zp[i] += 1e-5
            dpar = (vec_to_params(model, zp)[nme] - p[nme]) / 1e-5
            rel_std[nme] = float(np.sqrt(dpar**2 * cov[i, i]) / abs(p[nme]))
            at_bound[nme] = bool(min(abs(sol.x[i]-lo[i]), abs(sol.x[i]-hi[i])) < 0.05)
    except np.linalg.LinAlgError:
        rel_std = {n: np.nan for n in PARAM_NAMES[model]}; cond = np.inf
        at_bound = {n: False for n in PARAM_NAMES[model]}
    return dict(params=p, rel_std=rel_std, at_bound=at_bound, cond=cond, nparam=len(sol.x))

# Fit all three models on year 1. This takes only a few seconds in total, because the simulation
# loop is compiled with numba, so we simply do it here every time the notebook runs.
FITS = {m: fit(m) for m in ["A", "B", "C"]}

# quick report: training error at two horizons + number of parameters
for m in ["A", "B", "C"]:
    A, B, cols = matrices(m, FITS[m]["params"]); F, G = discretize(A, B, DT); ns = A.shape[0]
    U = U_of(cols, src_tr)
    print(f"Model {m}: 30-min RMSE = {_h_rmse(F,G,U,Ti_tr,2,len(Ti_tr),ns):.3f} K, "
          f"1-day RMSE = {_h_rmse(F,G,U,Ti_tr,96,len(Ti_tr),ns):.3f} K, "
          f"parameters = {FITS[m]['nparam']}")

Model A: 30-min RMSE = 0.739 K, 1-day RMSE = 2.027 K, parameters = 2
Model B: 30-min RMSE = 0.736 K, 1-day RMSE = 1.813 K, parameters = 4
Model C: 30-min RMSE = 0.712 K, 1-day RMSE = 1.462 K, parameters = 6


<a id="sec-params"></a>
## 7. Fitted parameters — are they physically sensible?

The table shows the fitted parameters in physical units, plus two numbers that are easy to sanity-check:

- **UA (W/K)** — the overall heat loss: how many watts the house loses per degree of indoor–outdoor difference.
- **time constant (h)** — roughly how many hours the building takes to react (its slowest "speed").

For a medium detached UK house we would expect **UA ≈ 150–350 W/K**, a slowest time constant of **about 1–3 days**, and
an effective solar window area of a few up to ~10 m².

In [10]:
def derived(model, p):
    A, _, _ = matrices(model, p)
    tau_h = -1.0 / np.max(np.linalg.eigvals(A).real) / 3600.0     # slowest time constant, in hours
    if   model == "A": UA = 1/p["Ri"]
    elif model == "B": UA = 1/(p["Rie"] + p["Rea"])
    else:              UA = 1/p["Ria"] + 1/(p["Rie"] + p["Rea"])
    return UA, tau_h

rows = []
for m in ["A", "B", "C"]:
    p = FITS[m]["params"]; UA, tau = derived(m, p)
    A, B, cols = matrices(m, p); F, G = discretize(A, B, DT); ns = A.shape[0]; U = U_of(cols, src_tr)
    row = {"Model": m, "UA [W/K]": round(UA, 1), "time const [h]": round(tau, 1),
           "1-day RMSE [K]": round(_h_rmse(F, G, U, Ti_tr, 96, len(Ti_tr), ns), 3), "# params": FITS[m]["nparam"]}
    for k, v in p.items():
        row[k] = (f"{v:.2e}" if (abs(v) > 1e3 or abs(v) < 1e-2) else round(v, 3))
    rows.append(row)
pd.DataFrame(rows).set_index("Model")

,UA [W/K],time const [h],1-day RMSE [K],# params,Ri,Ci,Rie,Rea,Ce,Ria,As
Model,,,,,,,,,,,
A,118.5,234.3,2.027,2,8.44e-03,1.00e+08,NaN,NaN,NaN,NaN,NaN
B,129.1,346.9,1.813,4,NaN,7.41e+07,7.01e-04,7.04e-03,1.00e+08,NaN,NaN
C,271.1,105.6,1.462,6,NaN,1.52e+07,5.00e-04,3.71e-03,1.00e+08,0.03,8.687


Read the **UA** and **time constant** columns first — those are the numbers the data really pin down, and they land
in the sensible range for all three models (Model C's UA is the most realistic). The individual R and C values are harder
to trust on their own, which is the subject of Section 9. Model C also finds a believable effective solar window area,
which A and B simply cannot represent.

<a id="sec-fit"></a>
## 8. Checking the fit on year 1

A realistic test of a forecasting model: every day we reset it to the measured indoor temperature and let it
predict the next **24 hours** from the inputs alone. We show one winter week and one summer week of the training year.

In [11]:
def sim_window(model, lo, hi, reinit=96):
    # 24-hour-ahead forecast: reset to the measured temperature every `reinit` steps (96 steps = 1 day)
    d = df[(df.year == 1) & (df.day_of_year >= lo) & (df.day_of_year < hi)]
    src = {"To": d.outside_temperature_C.values, "Qh": d.heating_power.values,
           "Qs": d.global_horizontal_solar_radiation.values}
    meas = d.indoor_temperature_C.values
    A, B, cols = matrices(model, FITS[model]["params"]); F, G = discretize(A, B, 900.0); ns = A.shape[0]
    U = U_of(cols, src); sim = np.empty(len(meas))
    for s in range(0, len(meas), reinit):
        e = min(s + reinit, len(meas))
        sim[s:e] = _freerun(F, G, np.ascontiguousarray(U[s:e]), np.full(ns, meas[s]), e - s, ns)[:, 0]
    x = d.day_of_year.values + d.second_of_day.values / 86400.0
    return x, meas, sim

COLM = {"A": "#9467bd", "B": "#2ca02c", "C": "#ff7f0e"}
fig = make_subplots(rows=1, cols=2, subplot_titles=("Winter week (days 15–22)", "Summer week (days 200–207)"))
for j, (lo, hi) in enumerate([(15, 22), (200, 207)], start=1):
    x, meas, _ = sim_window("A", lo, hi)
    fig.add_trace(go.Scatter(x=x, y=meas, line=dict(color="black", width=2.2), name="Measured",
                             showlegend=(j == 1), legendgroup="m"), 1, j)
    for m in ["A", "B", "C"]:
        x, _, sim = sim_window(m, lo, hi)
        fig.add_trace(go.Scatter(x=x, y=sim, line=dict(color=COLM[m], width=1.4), name=f"Model {m}",
                                 showlegend=(j == 1), legendgroup=m), 1, j)
    fig.update_xaxes(title_text="Day of year", row=1, col=j)
fig.update_yaxes(title_text="Indoor temperature [°C]", row=1, col=1)
fig.update_layout(title="24-hour-ahead forecast on year 1 (reset to measurement each day)")
save_fig(fig, "fig10_trainfit_windows", h=440)

All three models follow the measured line well in winter, where heating dominates. In **summer** the difference
shows: only **Model C** follows the daytime warm-up, because it is the only model with a sun term. Models A and B have no
way to "see" the sun, so they miss the summer peaks. (If instead we let the models run free for the whole week with no
daily reset, A and B drift away — the instability explained in Section 2 — which is why Question 3 measures error as a
function of how far ahead we forecast.)

<a id="sec-ident"></a>
## 9. Which parameters can we actually trust? (identifiability)

Fitting a model is one thing; knowing whether the data really *determine* each parameter is another. The bar chart
shows the estimated uncertainty of each parameter (smaller = better determined). A label "limit" means the parameter ran
into the edge of its allowed range while fitting — a clear sign the data could not decide its value.

In [12]:
fig = go.Figure()
for m in ["A", "B", "C"]:
    rs = FITS[m]["rel_std"]; ab = FITS[m]["at_bound"]
    fig.add_trace(go.Bar(x=[f"{m}:{k}" for k in rs], y=[rs[k]*100 for k in rs], marker_color=COLM[m],
                         name=f"Model {m}", text=["limit" if ab[k] else "" for k in rs], textposition="outside"))
fig.update_layout(title="Parameter uncertainty (log scale; 'limit' = ran into its allowed range)",
                  yaxis_title="relative uncertainty [%]", yaxis_type="log", barmode="group")
save_fig(fig, "fig11_identifiability", h=440)
fig.show()
# the condition number summarises how well-posed the whole fit is (small = good, huge = nearly unsolvable)
print("Fit conditioning (smaller is better):", {m: round(FITS[m]["cond"], 1) for m in ["A", "B", "C"]})
print("Parameters stuck at a limit:", {m: [k for k, v in FITS[m]["at_bound"].items() if v] for m in ["A","B","C"]})

Fit conditioning (smaller is better): {'A': 10.6, 'B': 119.9, 'C': 1433.1}
Parameters stuck at a limit: {'A': ['Ci'], 'B': ['Ce'], 'C': ['Rie', 'Ria', 'Ce']}


**What the data determine well:** the overall heat loss UA and the slowest time constant. These describe the big,
slow behaviour, which a full year of data constrains tightly. That is why Section 7 should be read through UA and the
time constant.

**What is shaky:**
- The **thermal capacitances** drift to the top of their allowed range. The data fix the *product* R×C (the speed) much
  better than C on its own, so the fit is almost indifferent to making C larger.
- The **split of mass between air and walls** (and of resistance between the inner and outer path) is only weakly
  determined: several different splits give almost the same indoor temperature.
- In **Model C**, several parameters run into the edge of their allowed range ($R_{ie}$, $R_{ia}$, $C_e$): the data
  cannot cleanly decide how the heat loss divides between the direct (window/ventilation) path and the wall path. The
  **solar window area $A_s$ itself is tightly determined** — sunshine has a strong, distinctive daily signature — which
  is reassuring, because it is exactly the term that gives Model C its edge.

<a id="sec-answer"></a>
## 10. Answer to Question 2

**We built three R-C models of increasing complexity and fitted them on year 1:**

- **Model A (1R1C)** — one lump, one resistance. Captures the overall heat loss but treats the whole house as a single
  fast body and has no sun term.
- **Model B (2R2C)** — adds a wall/mass lump, so it can show both the fast (air) and slow (wall) response seen in the
  data. Still no sun term.
- **Model C (3R2C)** — adds a direct window/ventilation path and a sun term. It is the only one that can reproduce
  summer solar warming.

**How they are built and fitted:** each is a small continuous-time R-C circuit, converted exactly to 15-minute steps with
the matrix exponential, and fitted on year 1 using multiple-shooting (short, repeatedly re-initialised forecasts) — which
is stable, unlike a naive free run, and matches the forecasting we test in Question 3.

**What the fit tells us:** the well-determined quantities (overall heat loss UA ≈ 120–270 W/K and a time constant of
several days) are physically reasonable for all three models. The detailed individual parameters are less certain:
capacitances run to their limits and, in Model C, so do some resistances — the data cannot cleanly split the heat loss
between the direct path and the wall path, even though the solar window area itself is pinned down well. So **Model C
fits the training year best, but its extra flexibility still carries a risk of overfitting**. Which model is genuinely
best is decided on year 2 in Question 3.